<a href="https://colab.research.google.com/github/fbeilstein/kai_programming/blob/master/python_vs_cupy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cupy-cuda12x

In [2]:

# =============================================================================
# GPU Computing with CuPy & Multiparticle Modelling
# Live-Coding Examples — Google Colab Edition
# =============================================================================
# Run this file in Google Colab. Enable GPU first:
#   Runtime → Change runtime type → T4 GPU
#
# Each exercise has:
#   PART A  – traditional Python loop (slow)
#   PART B  – NumPy vectorized (fast on CPU)
#   PART C  – CuPy / GPU (fast on GPU)
# =============================================================================

import time
import math
import numpy as np
import random

# Try importing CuPy — fall back gracefully if no GPU
try:
    import cupy as cp
    GPU_AVAILABLE = True
    try:
        dev = cp.cuda.runtime.getDeviceProperties(0)
        gpu_name = dev['name'].decode() if isinstance(dev['name'], bytes) else str(dev['name'])
    except Exception:
        gpu_name = "(unknown GPU)"
    print(f"✅ CuPy found. GPU: {gpu_name}")
except Exception as e:
    GPU_AVAILABLE = False
    print(f"⚠️  CuPy not available ({e}). GPU exercises will be skipped.")
    print("    In Colab: Runtime → Change runtime type → T4 GPU, then reinstall CuPy.")


def timer(label, fn, *args, n_warmup=0, **kwargs):
    """Run fn(*args, **kwargs), print timing, return result."""
    for _ in range(n_warmup):
        fn(*args, **kwargs)
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    # GPU sync before stopping the clock
    if GPU_AVAILABLE:
        cp.cuda.Stream.null.synchronize()
    t1 = time.perf_counter()
    ms = (t1 - t0) * 1000
    print(f"  {label:<40s}  {ms:>10.2f} ms")
    return result


def section(title):
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)

✅ CuPy found. GPU: Tesla T4


In [3]:

# =============================================================================
# EXERCISE 1: RANDOM WALK
# =============================================================================
# A particle takes N steps of ±1. What is its position over time?
# This is the foundational example from Rougier's book.
# =============================================================================
section("EXERCISE 1 — Random Walk (N=100_000 steps)")
N = 100_000

# ── PART A: Python loop ─────────────────────────────────────────────────────
def random_walk_loop(n):
    position = 0
    walk = [0] * n
    for i in range(n):
        step = 1 if random.random() > 0.5 else -1
        position += step
        walk[i] = position
    return walk

# ── PART B: NumPy ────────────────────────────────────────────────────────────
def random_walk_numpy(n):
    steps = np.where(np.random.random(n) > 0.5, 1, -1)
    return np.cumsum(steps)

# ── PART C: CuPy ─────────────────────────────────────────────────────────────
def random_walk_cupy(n):
    steps = cp.where(cp.random.random(n) > 0.5, 1, -1)
    result = cp.cumsum(steps)
    return cp.asnumpy(result)   # bring back for plotting

# ── BENCHMARKS ───────────────────────────────────────────────────────────────
walk_loop  = timer("Python loop", random_walk_loop, N)
walk_numpy = timer("NumPy cumsum", random_walk_numpy, N)
if GPU_AVAILABLE:
    walk_cupy = timer("CuPy  cumsum (GPU)", random_walk_cupy, N, n_warmup=1)

# ── SANITY CHECK ─────────────────────────────────────────────────────────────
print(f"\n  Final positions: loop={walk_loop[-1]}, numpy={walk_numpy[-1]}", end="")
if GPU_AVAILABLE:
    print(f", cupy={walk_cupy[-1]}", end="")
print("\n  (values differ — random seeds differ, that's fine)")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  DISCUSSION                                                              ║
# ║  • Loop: Python integer + list append per step — slow                   ║
# ║  • NumPy: generate ALL steps at once, then cumsum — one C call          ║
# ║  • CuPy:  identical API, runs on thousands of GPU threads               ║
# ║  • cumsum has a DATA DEPENDENCY (each value depends on the previous)    ║
# ║    → GPU advantage is modest here vs. embarrassingly parallel tasks     ║
# ╚══════════════════════════════════════════════════════════════════════════╝


  EXERCISE 1 — Random Walk (N=100_000 steps)
  Python loop                                   208.45 ms
  NumPy cumsum                                    9.75 ms
  CuPy  cumsum (GPU)                              0.65 ms

  Final positions: loop=-68, numpy=138, cupy=-76
  (values differ — random seeds differ, that's fine)


In [4]:
# =============================================================================
# EXERCISE 2: GAME OF LIFE (UNIFORM VECTORIZATION)
# =============================================================================
# Conway's Game of Life on a 2D grid. The goal: replace nested loops with
# array slicing to count neighbours and apply rules.
# =============================================================================
section("EXERCISE 2 — Game of Life (512×512 grid, 100 iterations)")
GRID = 512
STEPS = 100

# ── PART A: Python loop (count neighbours manually) ──────────────────────────
def gol_loop(Z, steps):
    Z = [row[:] for row in Z]   # shallow copy
    rows, cols = len(Z), len(Z[0])
    for _ in range(steps):
        N = [[0]*cols for _ in range(rows)]
        for r in range(1, rows-1):
            for c in range(1, cols-1):
                N[r][c] = (Z[r-1][c-1] + Z[r-1][c] + Z[r-1][c+1] +
                           Z[r  ][c-1]              + Z[r  ][c+1] +
                           Z[r+1][c-1] + Z[r+1][c] + Z[r+1][c+1])
        new_Z = [[0]*cols for _ in range(rows)]
        for r in range(1, rows-1):
            for c in range(1, cols-1):
                n = N[r][c]
                if Z[r][c] == 1:
                    new_Z[r][c] = 1 if 2 <= n <= 3 else 0
                else:
                    new_Z[r][c] = 1 if n == 3 else 0
        Z = new_Z
    return Z

# ── PART B: NumPy slicing ────────────────────────────────────────────────────
def gol_numpy(Z, steps):
    Z = Z.copy()
    for _ in range(steps):
        # Count all 8 neighbours simultaneously with array slices
        N = (Z[:-2,:-2] + Z[:-2,1:-1] + Z[:-2,2:] +
             Z[1:-1,:-2]               + Z[1:-1,2:] +
             Z[2:,:-2]  + Z[2:,1:-1]  + Z[2:,2:])
        # Apply rules with boolean masks (no loop!)
        birth   = (N == 3) & (Z[1:-1,1:-1] == 0)
        survive = ((N == 2) | (N == 3)) & (Z[1:-1,1:-1] == 1)
        Z[...] = 0
        Z[1:-1,1:-1][birth | survive] = 1
    return Z

# ── PART C: CuPy – IDENTICAL code, just cp instead of np ────────────────────
def gol_cupy(Z_gpu, steps):
    Z = Z_gpu.copy()
    for _ in range(steps):
        N = (Z[:-2,:-2] + Z[:-2,1:-1] + Z[:-2,2:] +
             Z[1:-1,:-2]               + Z[1:-1,2:] +
             Z[2:,:-2]  + Z[2:,1:-1]  + Z[2:,2:])
        birth   = (N == 3) & (Z[1:-1,1:-1] == 0)
        survive = ((N == 2) | (N == 3)) & (Z[1:-1,1:-1] == 1)
        Z[...] = 0
        Z[1:-1,1:-1][birth | survive] = 1
    return Z

# ── SETUP ────────────────────────────────────────────────────────────────────
np.random.seed(42)
grid_np = np.random.randint(0, 2, (GRID, GRID), dtype=np.uint8)

# Python loop only on a small grid (too slow for 512×512)
small = 32
grid_small = grid_np[:small, :small].tolist()
print(f"  (Python loop runs on {small}×{small} only — would take minutes on {GRID}×{GRID})")
timer(f"Python loop ({small}×{small}, {STEPS} steps)", gol_loop, grid_small, STEPS)

timer(f"NumPy  ({GRID}×{GRID}, {STEPS} steps)", gol_numpy, grid_np, STEPS)

if GPU_AVAILABLE:
    grid_gpu = cp.asarray(grid_np)
    timer(f"CuPy   ({GRID}×{GRID}, {STEPS} steps)", gol_cupy, grid_gpu, STEPS, n_warmup=1)

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  KEY INSIGHT                                                             ║
# ║  The NumPy trick: Z[:-2, :-2] is a VIEW of Z shifted by (-1,-1).       ║
# ║  Adding 8 such views counts all neighbours with no Python loop.         ║
# ║  CuPy: literally the same code — it dispatches to CUDA kernels.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝




  EXERCISE 2 — Game of Life (512×512 grid, 100 iterations)
  (Python loop runs on 32×32 only — would take minutes on 512×512)
  Python loop (32×32, 100 steps)                 53.56 ms
  NumPy  (512×512, 100 steps)                   180.96 ms
  CuPy   (512×512, 100 steps)                    54.35 ms


In [5]:
# =============================================================================
# EXERCISE 3: MANDELBROT SET (TEMPORAL VECTORIZATION)
# =============================================================================
# f(z) = z² + c. Count how many iterations before |z| > 2.
# Challenge: different pixels escape at different iterations.
# =============================================================================
section("EXERCISE 3 — Mandelbrot Set (800×600, max 100 iterations)")
XN, YN, MAXITER = 800, 600, 100

# ── PART A: Pure Python ───────────────────────────────────────────────────────
def mandelbrot_python(xmin, xmax, ymin, ymax, xn, yn, maxiter):
    result = [[0]*xn for _ in range(yn)]
    for j in range(yn):
        for i in range(xn):
            c = (xmin + i*(xmax-xmin)/xn) + 1j*(ymin + j*(ymax-ymin)/yn)
            z = 0j
            for n in range(maxiter):
                if abs(z) > 2.0:
                    result[j][i] = n
                    break
                z = z*z + c
    return result

# ── PART B: NumPy — vectorize ALL pixels simultaneously ──────────────────────
def mandelbrot_numpy(xmin, xmax, ymin, ymax, xn, yn, maxiter):
    X = np.linspace(xmin, xmax, xn, dtype=np.float32)
    Y = np.linspace(ymin, ymax, yn, dtype=np.float32)
    C = X[np.newaxis, :] + Y[:, np.newaxis] * 1j   # shape (yn, xn)
    Z = np.zeros_like(C)
    N = np.zeros(C.shape, dtype=np.int32)
    for n in range(maxiter):
        I = np.abs(Z) < 2.0       # mask: still iterating
        N[I] = n
        Z[I] = Z[I]**2 + C[I]    # update ONLY non-escaped pixels
    return N

# ── PART C: CuPy — same logic, GPU tensors ───────────────────────────────────
def mandelbrot_cupy(xmin, xmax, ymin, ymax, xn, yn, maxiter):
    X = cp.linspace(xmin, xmax, xn, dtype=cp.float32)
    Y = cp.linspace(ymin, ymax, yn, dtype=cp.float32)
    C = X[cp.newaxis, :] + Y[:, cp.newaxis] * 1j
    Z = cp.zeros_like(C)
    N = cp.zeros(C.shape, dtype=cp.int32)
    for n in range(maxiter):
        I = cp.abs(Z) < 2.0
        N[I] = n
        Z[I] = Z[I]**2 + C[I]
    return cp.asnumpy(N)

# ── BENCHMARKS ───────────────────────────────────────────────────────────────
bounds = (-2.5, 1.0, -1.25, 1.25)

# Python only on a tiny grid
print(f"  (Python loop: only 80×60 to save time)")
timer("Python loop (80×60, 100 iters)",
      mandelbrot_python, *bounds, 80, 60, MAXITER)

timer(f"NumPy  ({XN}×{YN}, {MAXITER} iters)",
      mandelbrot_numpy, *bounds, XN, YN, MAXITER)

if GPU_AVAILABLE:
    timer(f"CuPy   ({XN}×{YN}, {MAXITER} iters)",
          mandelbrot_cupy, *bounds, XN, YN, MAXITER, n_warmup=1)

# ── OPTIONAL: visualise result ───────────────────────────────────────────────
print("\n  To visualise:")
print("  import matplotlib.pyplot as plt")
print("  N = mandelbrot_numpy(*bounds, XN, YN, MAXITER)")
print("  plt.figure(figsize=(12,8))")
print("  plt.imshow(N, cmap='inferno'); plt.axis('off'); plt.show()")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  TEMPORAL VECTORIZATION LESSON                                           ║
# ║  • We still have a Python for-loop — but it's over ITERATIONS,          ║
# ║    not over pixels. All pixels are processed in parallel each iteration. ║
# ║  • The mask I shrinks each step: only un-converged pixels are updated.  ║
# ║  • On GPU: thousands of threads process all pixels in parallel.         ║
# ╚══════════════════════════════════════════════════════════════════════════╝



  EXERCISE 3 — Mandelbrot Set (800×600, max 100 iterations)
  (Python loop: only 80×60 to save time)
  Python loop (80×60, 100 iters)                 14.04 ms
  NumPy  (800×600, 100 iters)                   164.76 ms
  CuPy   (800×600, 100 iters)                   120.40 ms

  To visualise:
  import matplotlib.pyplot as plt
  N = mandelbrot_numpy(*bounds, XN, YN, MAXITER)
  plt.figure(figsize=(12,8))
  plt.imshow(N, cmap='inferno'); plt.axis('off'); plt.show()


In [6]:
# =============================================================================
# EXERCISE 4: PAIRWISE DISTANCES (SPATIAL VECTORIZATION)
# =============================================================================
# Compute all N×N pairwise distances between particles. Foundation of
# molecular dynamics and N-body simulations.
# =============================================================================
section("EXERCISE 4 — Pairwise Distances (N=2000 particles, 2D)")
N_PARTICLES = 2000

np.random.seed(0)
positions = np.random.rand(N_PARTICLES, 2).astype(np.float32)

# ── PART A: Python double loop ────────────────────────────────────────────────
def pairwise_loop(pos):
    n = len(pos)
    dist = [[0.0]*n for _ in range(n)]
    for i in range(n):
        for j in range(i+1, n):
            dx = pos[i][0] - pos[j][0]
            dy = pos[i][1] - pos[j][1]
            d = math.sqrt(dx*dx + dy*dy)
            dist[i][j] = d
            dist[j][i] = d
    return dist

# ── PART B: NumPy broadcasting ────────────────────────────────────────────────
def pairwise_numpy(pos):
    # pos shape: (N, 2)
    # pos[:, np.newaxis, :] shape: (N, 1, 2) — broadcast trick!
    diff = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]  # (N, N, 2)
    return np.sqrt((diff**2).sum(axis=-1))                 # (N, N)

# ── ALTERNATIVE: numpy outer subtraction (more explicit) ─────────────────────
def pairwise_numpy_v2(pos):
    dx = np.subtract.outer(pos[:, 0], pos[:, 0])   # (N, N)
    dy = np.subtract.outer(pos[:, 1], pos[:, 1])
    return np.hypot(dx, dy)                         # (N, N)

# ── PART C: CuPy ──────────────────────────────────────────────────────────────
def pairwise_cupy(pos_gpu):
    diff = pos_gpu[:, None, :] - pos_gpu[None, :, :]
    return cp.sqrt((diff**2).sum(axis=-1))

# ── BENCHMARKS ────────────────────────────────────────────────────────────────
small_pos = positions[:200].tolist()   # Python loop — tiny subset only
print(f"  (Python loop: only {len(small_pos)} particles to save time)")
timer(f"Python loop ({len(small_pos)} particles)", pairwise_loop, small_pos)

timer(f"NumPy broadcast ({N_PARTICLES} particles)", pairwise_numpy, positions)
timer(f"NumPy outer     ({N_PARTICLES} particles)", pairwise_numpy_v2, positions)

if GPU_AVAILABLE:
    pos_gpu = cp.asarray(positions)
    timer(f"CuPy broadcast  ({N_PARTICLES} particles)", pairwise_cupy, pos_gpu, n_warmup=1)

# ── DERIVE NEIGHBOURHOOD MASKS ────────────────────────────────────────────────
print("\n  Bonus: how to use the distance matrix for simulations")
print("  dist = pairwise_numpy(positions)          # (N, N)")
print("  neighbours   = dist < 0.05               # bool mask (N, N)")
print("  n_neighbours = neighbours.sum(axis=1)    # count per particle (N,)")
print("  close_pairs  = np.argwhere(neighbours)   # list of (i, j) pairs")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  LESSON: N² OPERATIONS THE NUMPY WAY                                    ║
# ║  • Loop: explicit O(N²) double loop — Python overhead per pair          ║
# ║  • NumPy: broadcasting creates (N, N, 2) array, then reduce along axis  ║
# ║  • CuPy: each (i,j) element computed by a separate GPU thread           ║
# ║  • Memory: N=10000 → 100M float32 = 400 MB — watch VRAM!               ║
# ╚══════════════════════════════════════════════════════════════════════════╝



  EXERCISE 4 — Pairwise Distances (N=2000 particles, 2D)
  (Python loop: only 200 particles to save time)
  Python loop (200 particles)                     4.20 ms
  NumPy broadcast (2000 particles)              138.15 ms
  NumPy outer     (2000 particles)               34.77 ms
  CuPy broadcast  (2000 particles)               49.13 ms

  Bonus: how to use the distance matrix for simulations
  dist = pairwise_numpy(positions)          # (N, N)
  neighbours   = dist < 0.05               # bool mask (N, N)
  n_neighbours = neighbours.sum(axis=1)    # count per particle (N,)
  close_pairs  = np.argwhere(neighbours)   # list of (i, j) pairs


In [7]:
# =============================================================================
# EXERCISE 5: LENNARD-JONES FORCES (MULTIPARTICLE SIMULATION)
# =============================================================================
# The core of molecular dynamics: compute forces between all particle pairs.
# V(r) = 4ε [(σ/r)¹² − (σ/r)⁶]
# =============================================================================
section("EXERCISE 5 — Lennard-Jones Forces (N=1000 particles, 3D)")
N_MD = 1000
BOX_L = 10.0
LJ_EPS = 1.0
LJ_SIGMA = 1.0
LJ_CUTOFF = 2.5   # r_cutoff = 2.5 σ (standard)
SOFTENING = 1e-6  # avoid division by zero

# --- Lattice initialization: place particles on a cubic grid to avoid overlaps ---
def make_lattice(n, box, sigma=1.0):
    """Place n particles on a simple cubic lattice with spacing sigma."""
    n_side = int(np.ceil(n**(1/3)))
    spacing = max(box / n_side, sigma * 1.05)
    pos = []
    for ix in range(n_side):
        for iy in range(n_side):
            for iz in range(n_side):
                pos.append([ix*spacing, iy*spacing, iz*spacing])
                if len(pos) == n:
                    break
            if len(pos) == n: break
        if len(pos) == n: break
    return np.array(pos, dtype=np.float32) % box

np.random.seed(42)
pos3d = make_lattice(N_MD, BOX_L)
pos3d_list = pos3d.tolist()

# ── PART A: Python triple loop ────────────────────────────────────────────────
def lj_forces_loop(positions, eps, sigma, L, cutoff):
    n = len(positions)
    forces = [[0.0, 0.0, 0.0] for _ in range(n)]
    for i in range(n):
        for j in range(i+1, n):
            # Minimum-image convention (periodic BC)
            dx = positions[i][0] - positions[j][0]
            dy = positions[i][1] - positions[j][1]
            dz = positions[i][2] - positions[j][2]
            dx -= L * round(dx / L)
            dy -= L * round(dy / L)
            dz -= L * round(dz / L)
            r2 = dx*dx + dy*dy + dz*dz
            if r2 < cutoff**2 and r2 > 0:
                sr2 = (sigma**2) / r2
                sr6 = sr2**3
                sr12 = sr6**2
                f_over_r = 24 * eps * (2*sr12 - sr6) / r2
                forces[i][0] += f_over_r * dx
                forces[i][1] += f_over_r * dy
                forces[i][2] += f_over_r * dz
                forces[j][0] -= f_over_r * dx
                forces[j][1] -= f_over_r * dy
                forces[j][2] -= f_over_r * dz
    return forces

# ── PART B: NumPy vectorized ─────────────────────────────────────────────────
def lj_forces_numpy(pos, eps, sigma, L, cutoff):
    # All pairwise vectors: shape (N, N, 3)
    r_vec = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]
    # Minimum-image (periodic BC)
    r_vec -= L * np.round(r_vec / L)
    # Squared distances
    r2 = (r_vec**2).sum(axis=-1)                  # (N, N)
    # Apply cutoff and avoid self-interaction
    mask = (r2 > SOFTENING) & (r2 < cutoff**2)
    r2_safe = np.where(mask, r2, 1.0)             # prevent division by zero
    # LJ force magnitude / r
    sr2 = (sigma**2) / r2_safe
    sr6 = sr2**3
    f_over_r = 24 * eps * (2*sr6**2 - sr6) / r2_safe
    f_over_r = np.where(mask, f_over_r, 0.0)
    # Force vectors: (N, N, 3), then sum over j axis
    F = (f_over_r[:, :, np.newaxis] * r_vec).sum(axis=1)  # (N, 3)
    return F

# ── PART C: CuPy ─────────────────────────────────────────────────────────────
def lj_forces_cupy(pos_gpu, eps, sigma, L, cutoff):
    r_vec = pos_gpu[:, None, :] - pos_gpu[None, :, :]
    r_vec -= L * cp.round(r_vec / L)
    r2 = (r_vec**2).sum(axis=-1)
    mask = (r2 > SOFTENING) & (r2 < cutoff**2)
    r2_safe = cp.where(mask, r2, 1.0)
    sr2 = (sigma**2) / r2_safe
    sr6 = sr2**3
    f_over_r = 24 * eps * (2*sr6**2 - sr6) / r2_safe
    f_over_r = cp.where(mask, f_over_r, 0.0)
    return (f_over_r[:, :, None] * r_vec).sum(axis=1)

# ── BENCHMARKS ────────────────────────────────────────────────────────────────
small_n = 100
print(f"  (Python loop: only {small_n} particles to save time)")
timer(f"Python loop ({small_n} particles)",
      lj_forces_loop, pos3d_list[:small_n], LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)

timer(f"NumPy  ({N_MD} particles)", lj_forces_numpy, pos3d, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)

if GPU_AVAILABLE:
    pos3d_gpu = cp.asarray(pos3d)
    timer(f"CuPy   ({N_MD} particles)",
          lj_forces_cupy, pos3d_gpu, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF, n_warmup=1)

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  PRODUCTION TIP: for N > 5000, the N×N matrix exceeds GPU VRAM.         ║
# ║  Use a VERLET NEIGHBOUR LIST: precompute which pairs are within          ║
# ║  r_cutoff + r_skin, and only update every ~20 steps.                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝



  EXERCISE 5 — Lennard-Jones Forces (N=1000 particles, 3D)
  (Python loop: only 100 particles to save time)
  Python loop (100 particles)                     6.34 ms
  NumPy  (1000 particles)                       123.15 ms
  CuPy   (1000 particles)                        13.80 ms


In [8]:

# =============================================================================
# EXERCISE 6: VELOCITY VERLET INTEGRATOR — FULL MD SIMULATION STEP
# =============================================================================
# Putting it all together: one timestep of molecular dynamics on the GPU.
# =============================================================================
section("EXERCISE 6 — MD Integrator: 100 steps, N=500 particles (numpy vs cupy)")
N_SIM = 500
DT = 0.002      # Small timestep — critical for stability!

np.random.seed(7)
pos_sim = make_lattice(N_SIM, BOX_L)  # lattice — no overlaps!
vel_sim = np.random.randn(N_SIM, 3).astype(np.float32) * 0.3  # cold start
vel_sim -= vel_sim.mean(axis=0)  # zero center-of-mass velocity

# Shared helper ───────────────────────────────────────────────────────────────
def total_energy_numpy(pos, vel, eps, sigma, L, cutoff):
    KE = 0.5 * (vel**2).sum()
    # Potential — use forces function (hacky but works for demo)
    r_vec = pos[:, np.newaxis, :] - pos[np.newaxis, :, :]
    r_vec -= L * np.round(r_vec / L)
    r2 = (r_vec**2).sum(axis=-1)
    mask = (r2 > SOFTENING) & (r2 < cutoff**2)
    r2_safe = np.where(mask, r2, 1.0)
    sr6 = ((sigma**2) / r2_safe)**3
    PE = (4*eps * (sr6**2 - sr6) * mask).sum() * 0.5  # ×0.5: each pair counted twice
    return float(KE), float(PE)

# ── NumPy simulation ──────────────────────────────────────────────────────────
def run_md_numpy(pos, vel, steps=100):
    pos = pos.copy(); vel = vel.copy()
    energies = []
    for step in range(steps):
        F = lj_forces_numpy(pos, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)
        vel += 0.5 * F * DT       # half-kick
        pos += vel * DT           # drift
        pos %= BOX_L              # periodic BC (wrap)
        F = lj_forces_numpy(pos, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)
        vel += 0.5 * F * DT       # half-kick
        if step % 10 == 0:
            KE, PE = total_energy_numpy(pos, vel, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)
            energies.append((step, KE, PE, KE+PE))
    return pos, vel, energies

# ── CuPy simulation ───────────────────────────────────────────────────────────
def run_md_cupy(pos_gpu, vel_gpu, steps=100):
    pos = pos_gpu.copy(); vel = vel_gpu.copy()
    energies = []
    for step in range(steps):
        F = lj_forces_cupy(pos, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)
        vel += 0.5 * F * DT
        pos += vel * DT
        pos %= BOX_L
        F = lj_forces_cupy(pos, LJ_EPS, LJ_SIGMA, BOX_L, LJ_CUTOFF)
        vel += 0.5 * F * DT
        if step % 10 == 0:
            KE = float(0.5 * (vel**2).sum())
            energies.append((step, KE))   # simplified
    return cp.asnumpy(pos), cp.asnumpy(vel), energies

# ── BENCHMARKS ────────────────────────────────────────────────────────────────
STEPS_SIM = 100
print(f"  Running {STEPS_SIM} steps of velocity-Verlet MD on {N_SIM} particles...")

_, _, energies_np = timer(
    f"NumPy MD  ({N_SIM} particles, {STEPS_SIM} steps)",
    run_md_numpy, pos_sim, vel_sim, STEPS_SIM)

if GPU_AVAILABLE:
    pos_sim_gpu = cp.asarray(pos_sim)
    vel_sim_gpu = cp.asarray(vel_sim)
    _, _, energies_cp = timer(
        f"CuPy  MD  ({N_SIM} particles, {STEPS_SIM} steps)",
        run_md_cupy, pos_sim_gpu, vel_sim_gpu, STEPS_SIM, n_warmup=0)

# ── PRINT ENERGY TABLE ────────────────────────────────────────────────────────
print("\n  Energy conservation check (NumPy run):")
print(f"  {'Step':>6}  {'KE':>10}  {'PE':>10}  {'Total E':>12}")
for step, KE, PE, E_tot in energies_np:
    print(f"  {step:>6}  {KE:>10.3f}  {PE:>10.3f}  {E_tot:>12.3f}")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  WHAT TO LOOK FOR                                                        ║
# ║  • Total energy (KE + PE) should be roughly conserved (±1-2%)          ║
# ║    if DT is small enough. Drift = numerical instability.                ║
# ║  • CuPy speedup grows with N. Try N=2000, 5000!                        ║
# ║  • Real MD uses r_cutoff + Verlet neighbour list to avoid O(N²).       ║
# ╚══════════════════════════════════════════════════════════════════════════╝



  EXERCISE 6 — MD Integrator: 100 steps, N=500 particles (numpy vs cupy)
  Running 100 steps of velocity-Verlet MD on 500 particles...
  NumPy MD  (500 particles, 100 steps)         4178.12 ms
  CuPy  MD  (500 particles, 100 steps)          969.84 ms

  Energy conservation check (NumPy run):
    Step          KE          PE       Total E
       0      64.874   -1603.587     -1538.713
      10      64.982   -1603.663     -1538.681
      20      65.163   -1603.827     -1538.664
      30      65.340   -1603.988     -1538.648
      40      65.387   -1604.035     -1538.648
      50      65.136   -1603.768     -1538.632
      60      64.414   -1603.046     -1538.632
      70      63.122   -1601.754     -1538.632
      80      61.353   -1599.953     -1538.599
      90      59.468   -1598.002     -1538.534


In [9]:

# =============================================================================
# EXERCISE 7: BONUS — GRAY-SCOTT REACTION DIFFUSION ON GPU
# =============================================================================
# Two chemicals U and V diffuse and react. Produces stunning patterns.
# This shows how PDEs map perfectly to GPU arrays.
# =============================================================================
section("EXERCISE 7 (Bonus) — Gray-Scott PDE (512×512, 500 steps)")
GS_N = 512
GS_STEPS = 500
# Bacteria parameters:
Du, Dv, f, k = 0.16, 0.08, 0.035, 0.065
dt_gs = 1.0

def laplacian_np(Z):
    """5-point Laplacian stencil with periodic BC."""
    return (np.roll(Z,  1, axis=0) + np.roll(Z, -1, axis=0) +
            np.roll(Z,  1, axis=1) + np.roll(Z, -1, axis=1) - 4*Z)

def gray_scott_numpy(N, steps):
    U = np.ones((N, N), dtype=np.float32)
    V = np.zeros((N, N), dtype=np.float32)
    # Seed V in the centre
    r = N // 10
    U[N//2-r:N//2+r, N//2-r:N//2+r] = 0.5
    V[N//2-r:N//2+r, N//2-r:N//2+r] = 0.25
    U += 0.01 * np.random.randn(N, N).astype(np.float32)
    V += 0.01 * np.random.randn(N, N).astype(np.float32)
    for _ in range(steps):
        uvv = U * V * V
        U += dt_gs * (Du * laplacian_np(U) - uvv + f*(1 - U))
        V += dt_gs * (Dv * laplacian_np(V) + uvv - (f + k)*V)
        np.clip(U, 0, 1, out=U)
        np.clip(V, 0, 1, out=V)
    return U, V

if GPU_AVAILABLE:
    def laplacian_cp(Z):
        return (cp.roll(Z,  1, axis=0) + cp.roll(Z, -1, axis=0) +
                cp.roll(Z,  1, axis=1) + cp.roll(Z, -1, axis=1) - 4*Z)

    def gray_scott_cupy(N, steps):
        U = cp.ones((N, N), dtype=cp.float32)
        V = cp.zeros((N, N), dtype=cp.float32)
        r = N // 10
        U[N//2-r:N//2+r, N//2-r:N//2+r] = 0.5
        V[N//2-r:N//2+r, N//2-r:N//2+r] = 0.25
        U += 0.01 * cp.random.randn(N, N).astype(cp.float32)
        V += 0.01 * cp.random.randn(N, N).astype(cp.float32)
        for _ in range(steps):
            uvv = U * V * V
            U += dt_gs * (Du * laplacian_cp(U) - uvv + f*(1 - U))
            V += dt_gs * (Dv * laplacian_cp(V) + uvv - (f + k)*V)
            cp.clip(U, 0, 1, out=U)
            cp.clip(V, 0, 1, out=V)
        return cp.asnumpy(U), cp.asnumpy(V)

np.random.seed(1)
timer(f"NumPy Gray-Scott ({GS_N}×{GS_N}, {GS_STEPS} steps)",
      gray_scott_numpy, GS_N, GS_STEPS)

if GPU_AVAILABLE:
    timer(f"CuPy  Gray-Scott ({GS_N}×{GS_N}, {GS_STEPS} steps)",
          gray_scott_cupy, GS_N, GS_STEPS, n_warmup=1)

print("\n  To visualise the patterns:")
print("  U, V = gray_scott_numpy(512, 2000)   # or gray_scott_cupy")
print("  plt.figure(figsize=(10,10))")
print("  plt.imshow(V, cmap='RdYlBu'); plt.axis('off')")
print("  plt.title('Gray-Scott V (bacteria pattern)'); plt.show()")

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  LESSON: PDE → ARRAY STENCIL                                             ║
# ║  • The Laplacian ∇²U is just a sum of 4 shifted views of U             ║
# ║  • np.roll gives periodic boundary conditions for free                  ║
# ║  • Replacing np with cp gives GPU version — literally 2 characters      ║
# ║  • Try different (f, k) values — see the table in the lecture slides!   ║
# ╚══════════════════════════════════════════════════════════════════════════╝



  EXERCISE 7 (Bonus) — Gray-Scott PDE (512×512, 500 steps)
  NumPy Gray-Scott (512×512, 500 steps)        2338.57 ms
  CuPy  Gray-Scott (512×512, 500 steps)         547.45 ms

  To visualise the patterns:
  U, V = gray_scott_numpy(512, 2000)   # or gray_scott_cupy
  plt.figure(figsize=(10,10))
  plt.imshow(V, cmap='RdYlBu'); plt.axis('off')
  plt.title('Gray-Scott V (bacteria pattern)'); plt.show()
